In [ ]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from typing import Tuple, List
import io
import re
import numpy as np
import requests
import time

# Load Datasets

In [223]:
from typing import Tuple, List
import io
import re
import numpy as np
import requests
import time

def days_and_hours_since_2020(minutes_since_epoch) -> Tuple[int, int]:
    # Decode minutes_since_epoch → (year, month, day, hour)
    total_days, rem = divmod(int(minutes_since_epoch), 24 * 60)
    hour = rem // 60

    def _days_since_year0(y, m, d):
        days = [0, 31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
        if (y % 4 == 0 and y % 100 != 0) or (y % 400 == 0):
            days[2] = 29
        return y * 365 + y // 4 - y // 100 + y // 400 + sum(days[1:m]) + (d - 1)

    abs_days = total_days + _days_since_year0(1970, 1, 1)

    year = int(abs_days / 365.2425)
    while _days_since_year0(year + 1, 1, 1) <= abs_days:
        year += 1
    while _days_since_year0(year, 1, 1) > abs_days:
        year -= 1

    month = 1
    while month < 12 and _days_since_year0(year, month + 1, 1) <= abs_days:
        month += 1

    day = abs_days - _days_since_year0(year, month, 1) + 1

    # Day-of-year in 365-day scheme (Feb always = 28 days, Feb 29 excluded)
    days_per_month = [0, 31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    doy = sum(days_per_month[1:month]) + (day - 1)   # 0-indexed

    day_index = (year - 2020) * 365 + doy
    return (day_index, hour)

# Start with the Openmeteo data, then load the BSRN data.
NGROK_URL = "https://curtly-salvageable-clementina.ngrok-free.dev"

# Placeholder for all data
class MeteoDataChunked:
    def _encode_days_and_hours(self):
        self.h_cos = np.zeros(len(self.timestamps))
        self.h_sin = np.zeros(len(self.timestamps))
        self.d_cos = np.zeros(len(self.timestamps))
        self.d_sin = np.zeros(len(self.timestamps))
        for i, t in enumerate(self.timestamps):
            d, h = days_and_hours_since_2020(t)
            self.h_cos[i] = np.cos(h/24 * 2 * np.pi) * np.sqrt(2)
            self.h_sin[i] = np.sin(h/24 * 2 * np.pi) * np.sqrt(2)
            self.d_cos[i] = np.cos(d/365 * 2 * np.pi) * np.sqrt(2)
            self.d_sin[i] = np.sin(d/365 * 2 * np.pi) * np.sqrt(2)

    def __init__(self, name, timestamps: np.ndarray, global_rad: np.ndarray,
                temp: np.ndarray, rel_hum: np.ndarray, pressure: np.ndarray, wind_speed: np.ndarray, means=None, sigmas=None):
        self.name       = name
        self.timestamps = timestamps
        if means is None or sigmas is None:
            self.global_rad = global_rad
            self.temp       = temp
            self.pressure   = pressure
            self.rel_hum    = rel_hum
            self.wind_speed = wind_speed
        else:
            self.global_rad = (global_rad - means[0])/sigmas[0]
            self.temp       = (temp - means[1])/sigmas[1]
            self.pressure   = (pressure - means[2])/sigmas[2]
            self.rel_hum    = (rel_hum - means[3])/sigmas[3]
            self.wind_speed = wind_speed # TODO
        self._encode_days_and_hours()

    @classmethod
    def from_meteodata(cls, md: MeteoDataChunked):
        return MeteoDataChunked(md.name, md.timestamps, md.global_rad, md.temp, md.rel_hum,
                                md.pressure, md.wind_speed, md.means, md.sigmas)

    def __repr__(self):
        start = days_and_hours_since_2020(self.timestamps[0])
        end   = days_and_hours_since_2020(self.timestamps[-1])
        ret = f"Timestamps since {start[0]} to {end[0]}."
        return ret

    def __len__(self):
        return len(self.timestamps)

    def cut(self, start_idx: int, end_idx: int) -> MeteoDataChunked:
        if start_idx >= end_idx:
            return None
        new = MeteoDataChunked(
            self.name,
            self.timestamps[start_idx:end_idx],
            self.global_rad[start_idx:end_idx],
            self.temp[start_idx:end_idx],
            self.rel_hum[start_idx:end_idx],
            self.pressure[start_idx:end_idx],
            self.wind_speed[start_idx:end_idx],
        )
        return new

    @staticmethod
    def stats_from_more_chunks(chunks: List['MeteoDataChunked']) -> Tuple[np.ndarray, np.ndarray]:
        new = MeteoDataChunked(
            "NORMALIZED",
            np.concatenate([c.timestamps  for c in chunks]),
            np.concatenate([c.global_rad  for c in chunks]),
            np.concatenate([c.temp        for c in chunks]),
            np.concatenate([c.rel_hum     for c in chunks]),
            np.concatenate([c.pressure    for c in chunks]),
            np.concatenate([c.wind_speed  for c in chunks]),
        )
        mean = np.array([
            new.global_rad.mean(),
            new.temp.mean(),
            new.pressure.mean(),
            new.rel_hum.mean(),
        ])
        std = np.array([
            new.global_rad.std(),
            new.temp.std(),
            new.pressure.std(),
            new.rel_hum.std(),
        ])
        return mean, std
        

In [224]:
# Connect to the ngrok URL and retrieve data from dataset-openmeteo/saved
def load_openmeteo_data() -> List[MeteoDataChunked]:
    ret = []
    cities = ["Dunakezsi", "Pecel", "Szigetszentmiklos", "Vecses"]
    for c in cities:
        response = requests.get(f"{NGROK_URL}/dataset-openmeteo/saved/{c}.npz")
        data = np.load(io.BytesIO(response.content))
        ret.append(MeteoDataChunked(c, data["timestamps"],    data["global_tilted_irradiance"],
                                    data["temperature_2m"],   data["relative_humidity_2m"],
                                    data["surface_pressure"], data["wind_speed_10m"]))
    return ret

def load_bsrn_data() -> List[MeteoDataChunked]:
    base_url = f"{NGROK_URL}/dataset-bsrn/saved"

    def list_files() -> List[str]:
        html = requests.get(base_url).text
        return re.findall(r'href="([^"/?][^"]*)"', html)

    ret = []
    files = list_files()
    for f in files:
        if f == "dates.txt":
            continue
        response = requests.get(f"{base_url}/{f}")
        data = np.load(io.BytesIO(response.content))
        # Get the chunk number rn
        number = int((f[5:])[0:-4])
        # Get the budapest wind speed
        response = requests.get(f"{NGROK_URL}/dataset-openmeteo/saved/Budapest{number}.npz")
        data_openmeteo = np.load(io.BytesIO(response.content))
        wind_speed = data_openmeteo["wind_speed_10m"]

        ret.append(MeteoDataChunked("BSRN", data["timestamps"],  data["global_rad"],
                                            data["temperature"], data["humidity"],
                                            data["pressure"],    wind_speed))
        # Stupid ngrok max reqs hack
        time.sleep(1)
    
    return ret

openmeteo_chunks = load_openmeteo_data()
bsrn_chunks      = load_bsrn_data()

# Data Loaded, create PyTorch Dataset

In [252]:
# The sliding window length
CONSECUTIVE_HOURS = 3 * 24 # 3 days
TEST_FRACTION     = 0.20

# Create train and testing data. Merge them and compute mean and stds.
# First, process the openmeteo data. BSRN will be used for tinetuning.

def create_train_and_test_chunks(chunks, batch_size: int = 32):
    """
    TODO
    """
    n = len(chunks)
    TRAIN_TO_TEST = int(1/TEST_FRACTION)

    all_train_data,   all_train_labels = [], []
    all_test_data,    all_test_labels  = [], []

    train_data = []
    test_data  = []

    # Perform cross validation on more chunks. From this, compute train statistics (mean, std).
    for i, c in enumerate(chunks):
        L = len(c.timestamps)
        W = CONSECUTIVE_HOURS
        if len(c.timestamps)/TRAIN_TO_TEST >= CONSECUTIVE_HOURS:
            # Slice the data first.
            test_start = int((1.0 - ((i % TRAIN_TO_TEST) + 1) * TEST_FRACTION) * L)
            test_end   = int((1.0 -  (i % TRAIN_TO_TEST)      * TEST_FRACTION) * L)
            # Slice before
            before = c.cut(0, test_start)
            if before is not None:
                train_data.append(c.cut(0, test_start))
            # Slide test data
            between = c.cut(test_start, test_end)
            if between is not None:
                test_data.append(c.cut(test_start, test_end))
            # Slide train data after
            after = c.cut(test_end, L)
            if after is not None:
                train_data.append(after)
        else:
            # The testing data would be too small. Just append everything to training
            train_data.append(c)

    meteo_train_means, meteo_train_sigmas = MeteoDataChunked.stats_from_more_chunks(train_data)
    print(f"Training data computed means: {meteo_train_means}, sigmas: {meteo_train_sigmas}")
    
    # Perform normalization
    train_data_normalized = []; test_data_normalized = []
    for c in train_data:
        c.means  = meteo_train_means
        c.sigmas = meteo_train_sigmas
        train_data_normalized.append(MeteoDataChunked.from_meteodata(c))
    for c in test_data:
        c.means  = meteo_train_means
        c.sigmas = meteo_train_sigmas
        test_data_normalized.append(MeteoDataChunked.from_meteodata(c))

    return train_data_normalized, test_data_normalized

openmeteo_train_chunks, openmeteo_test_chunks = create_train_and_test_chunks(openmeteo_chunks)
bsrn_train_chunks, bsrn_test_chunks = create_train_and_test_chunks(bsrn_chunks)


Training data computed means: [ 181.51119527   11.98794258 1001.99537615   71.43260004], sigmas: [271.15680132   9.07571069   8.22160975  19.33177785]
Training data computed means: [ 145.29616277   11.6423176  1000.26299884   69.08875456], sigmas: [230.00696723   8.89172246   8.18279375  21.24488872]


In [303]:
# Test the data is correct, actually

#plt.plot(bsrn_train_chunks[0].timestamps)
#plt.show()

In [ ]:
# Now, create sliding windows from such data and data PyTorch data loaders
def create_dataloader(chunks_normalized: List[MeteoDataChunked], cons_hours, batch_size=32, shuffle=True):
    """
    Create sliding windows from every chunk from chunks_normalized.
    Returns (data: (N, cons_hours, 8), labels: (N, cons_hours)),
    where N is the number of sliding windows created
    """
    N_FEATURES = 8       # cos(h), sin(h), cos(d), sin(d), GTI(i-1), temp, pressure, hum

    N = 0
    for c in chunks_normalized:
        N += len(c.timestamps) - cons_hours

    data   = np.zeros((N, cons_hours, N_FEATURES), dtype=np.float32)
    labels = np.zeros((N, cons_hours),             dtype=np.float32)

    i = 0
    for c in chunks_normalized:
        for j in range(1, len(c.timestamps) - cons_hours + 1):
            data[i, :, :] = np.vstack([
                c.h_cos[j:j+cons_hours],
                c.h_sin[j:j+cons_hours],
                c.d_cos[j:j+cons_hours],
                c.d_sin[j:j+cons_hours],
                c.global_rad[j-1:j-1+cons_hours],
                c.temp[j:j+cons_hours],
                c.pressure[j:j+cons_hours],
                c.rel_hum[j:j+cons_hours],
            ]).T
            labels[i, :] = c.global_rad[j:j+cons_hours]
            i += 1

    ds = TensorDataset(torch.from_numpy(data), torch.from_numpy(labels))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=shuffle)
    return loader

print("Creating dataloaders")
openmeteo_train_loader = create_dataloader(openmeteo_train_chunks, CONSECUTIVE_HOURS)
print("Openmeteo train loader created!")
openmeteo_test_loader  = create_dataloader(openmeteo_test_chunks,  CONSECUTIVE_HOURS, shuffle=False)
print("Openmeteo test loader created!")
bsrn_train_loader      = create_dataloader(bsrn_train_chunks, CONSECUTIVE_HOURS)
print("BSRN train loader created!")
bsrn_test_loader       = create_dataloader(bsrn_test_chunks, CONSECUTIVE_HOURS, shuffle=False)
print("BSRN test loader created!")

Creating dataloaders
Openmeteo train loader created!
Openmeteo test loader created!
BSRN train loader created!
BSRN test loader created!
